# EE200 Project Q2
## The Midnight Episode: Catching the Arrhythmia

**Complete Submission Style Notebook**

## Introduction
Electrocardiogram (ECG) signals represent the electrical activity of the heart. In a healthy heart, ECG patterns repeat periodically and consist of P-wave, QRS complex and T-wave. During arrhythmia, this regular pattern is disturbed.

### Objectives
- Understand ECG characteristics
- Analyze healthy ECG in time and frequency domains
- Extract a heartbeat template
- Perform normalized correlation
- Detect arrhythmia onset
- Verify results using a spectrogram
- Study sampling and aliasing effects


# Part (a) Reading the Signal

### Given
- Sampling frequency = 250 Hz
- Number of samples = 5000
- Beat period = 0.8 s

### Theory
Sampling frequency defines how many samples are recorded every second. Recording duration is determined from the number of samples and sampling frequency.

### Derivation
T = N/fs

T = 5000/250 = 20 seconds

Heart Rate = 60/0.8 = 75 BPM

Samples per beat = 0.8 × 250 = 200 samples

Fundamental frequency = 1/0.8 = 1.25 Hz

### Interpretation
The ECG recording contains 25 heartbeats approximately.


# Part (a) Verification Code

In [ ]:
fs = 250
N = 5000
beat_period = 0.8

duration = N/fs
heart_rate = 60/beat_period
samples_per_beat = beat_period*fs
f0 = 1/beat_period

print("Duration =", duration)
print("Heart Rate =", heart_rate)
print("Samples/Beat =", samples_per_beat)
print("Fundamental Frequency =", f0)


# Part (b) Healthy Heart in Frequency Domain

## Theory

A periodic signal produces discrete harmonics in the frequency domain.

### ECG Components
- P wave → low frequency
- QRS complex → high frequency
- T wave → low frequency

### Why?
Sharp transitions require high-frequency components.

The QRS complex is the sharpest feature of ECG and therefore dominates higher frequencies.

### If Heart Rate Increases
For 150 BPM:

f0 = 150/60 = 2.5 Hz

All harmonics become more widely spaced.


## Expected Spectrum Observations

Healthy ECG:
- Distinct harmonics
- Stable spacing
- Dominant low-frequency energy

Abnormal ECG:
- Harmonic distortion
- Irregular frequency content


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

ecg_signal = np.load("patient_ecg.npy")
template = np.load("template.npy")

X = np.fft.fft(ecg_signal)
freq = np.fft.fftfreq(len(ecg_signal), d=1/250)

plt.figure(figsize=(10,4))
plt.plot(freq[:len(freq)//2], np.abs(X[:len(X)//2]))
plt.title("Magnitude Spectrum of ECG")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.grid(True)
plt.show()


# Part (c) Cutting a Heartbeat (Windowing)

## Theory

Windowing extracts a small portion of a signal.

### Goal
Extract exactly one healthy heartbeat.

Required width:
200 samples

### Case 1: 80 Samples
Problems:
- Incomplete beat
- Missing P/T waves
- Poor template

### Case 2: 600 Samples
Problems:
- Multiple beats included
- Reduced uniqueness

## Time-Frequency Tradeoff

Short window:
- Better time resolution
- Poor frequency resolution

Long window:
- Better frequency resolution
- Poor time resolution


In [ ]:
plt.figure(figsize=(10,3))
plt.plot(template)
plt.title("Extracted Healthy Beat Template")
plt.grid(True)
plt.show()


# Part (d) Template Matching using Correlation

## Theory

Correlation measures similarity between two signals.

### Formula

rho = dot(template, segment) / (||template|| ||segment||)

### Range

-1 ≤ rho ≤ 1

### Interpretation

rho ≈ 1 → Healthy beat

rho ≈ 0 → Different beat

rho ≈ -1 → Inverted beat

### Importance of Normalization

Normalization removes amplitude dependency.


In [ ]:
def normalized_correlation(template, segment):

    numerator = np.dot(template, segment)

    denominator = (
        np.linalg.norm(template)
        * np.linalg.norm(segment)
    )

    if denominator == 0:
        return 0

    return numerator / denominator


## Beat-by-Beat Analysis

The assignment specifically asks for beat-by-beat comparison.

Therefore the sliding step is equal to template length.


In [ ]:
L = len(template)

rho_values = []
indices = []

for m in range(0, len(ecg_signal)-L+1, L):

    segment = ecg_signal[m:m+L]

    rho = normalized_correlation(
        template,
        segment
    )

    rho_values.append(rho)
    indices.append(m)


## Correlation Plot Analysis

Healthy beats should produce correlation values close to +1.

Arrhythmic beats should produce lower values.


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(indices, rho_values, marker='o')
plt.axhline(0.5, linestyle='--')
plt.xlabel("Starting Sample")
plt.ylabel("Correlation")
plt.title("Beat-by-Beat Correlation")
plt.grid(True)
plt.show()


# Part (e) Onset Detection

## Theory

Arrhythmia onset is defined as the first beat whose correlation falls below a predefined threshold.

### Threshold Choice

0.5

### High Threshold
- False positives

### Low Threshold
- Missed arrhythmia

A balance is required.


In [ ]:
def find_onset(ecg_signal, template, threshold=0.5):

    L = len(template)

    for m in range(0, len(ecg_signal)-L+1, L):

        segment = ecg_signal[m:m+L]

        rho = normalized_correlation(template, segment)

        if rho < threshold:
            return m

    return -1


In [ ]:
onset = find_onset(ecg_signal, template)

print("Onset Sample =", onset)
print("Onset Time =", onset/250 if onset!=-1 else "Not Found")


In [ ]:
time = np.arange(len(ecg_signal))/250

plt.figure(figsize=(12,4))
plt.plot(time, ecg_signal)

if onset != -1:
    plt.axvline(onset/250, linestyle='--')

plt.title("Detected Arrhythmia Onset")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()


# Spectrogram Analysis

## Theory

A spectrogram is a time-frequency representation generated using the Short Time Fourier Transform (STFT).

### Why Spectrogram?

A normal FFT only shows which frequencies exist.

A spectrogram shows:
- Which frequencies exist
- When they exist

### Window Length Selection

nperseg = 200

Reason:
One heartbeat occupies approximately 200 samples.


In [ ]:
f, t, Sxx = signal.spectrogram(
    ecg_signal,
    fs=250,
    nperseg=200
)

plt.figure(figsize=(12,5))
plt.pcolormesh(t, f, 10*np.log10(Sxx+1e-10))
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("ECG Spectrogram")
plt.colorbar(label="Power (dB)")
plt.show()


# Part (f) Sampling and Aliasing

## Theory

Maximum ECG frequency ≈ 40 Hz

### Nyquist Criterion

fs > 2 × fmax

fs > 80 Hz

### If fs = 50 Hz

Aliasing occurs.

High-frequency information folds into lower frequencies.

### Consequences

- Distorted QRS complexes
- Incorrect detection
- Reduced diagnostic accuracy

### Solution

Apply an anti-aliasing low-pass filter before downsampling.


# Final Results Discussion

After running the dataset, discuss:

1. Average healthy correlation score
2. Onset location
3. Spectrogram changes
4. Detector accuracy
5. Strengths and limitations

Include screenshots of all generated figures.


# Conclusion

A complete arrhythmia detection system was developed using template matching and normalized correlation. Healthy beats produced high similarity scores, whereas abnormal beats caused a significant reduction in correlation. Spectrogram analysis provided additional confirmation in the frequency domain. The combined approach successfully identifies the onset of abnormal cardiac activity.
